# Week 6 -- Function 8: GP + UCB Exploitation (8D)

In [1]:
# -- WEEKLY OBSERVATIONS LOG
import numpy as np

INITIAL_SIZE = 40
DATA_PATH_X  = '../data/function_8/initial_inputs.npy'
DATA_PATH_Y  = '../data/function_8/initial_outputs.npy'

weekly_log = [
    ([0.816815, 0.656459, 0.258882, 0.854757, 0.737459, 0.233756, 0.838932, 0.820863], 7.2751498504466),        # W1: 7.275
    ([0.856208, 0.914280, 0.313812, 0.369507, 0.710367, 0.257166, 0.646775, 0.025767], 7.627455582011601),      # W2: 7.627
    ([0.076274, 0.101214, 0.383035, 0.338493, 0.113685, 0.882235, 0.615428, 0.796463], 9.0382459830856),        # W3: 9.038
    ([0.000000, 0.000000, 0.000000, 0.000000, 1.000000, 1.000000, 0.000000, 1.000000], 9.5183),                 # W4: 9.518 BEST weekly
    ([0.076274, 0.101214, 0.383035, 0.338493, 0.113685, 0.882235, 0.615428, 0.796463], 9.0382459830856),        # W5: 9.038 (W3 duplicate -- wasted)
]

X_disk = np.load(DATA_PATH_X)
Y_disk = np.load(DATA_PATH_Y)

n_missing = (INITIAL_SIZE + len(weekly_log)) - len(Y_disk)
if n_missing > 0:
    new_entries = weekly_log[len(weekly_log) - n_missing:]
    for x_vals, y_val in new_entries:
        X_disk = np.vstack([X_disk, np.array([x_vals])])
        Y_disk = np.append(Y_disk, y_val)
    np.save(DATA_PATH_X, X_disk)
    np.save(DATA_PATH_Y, Y_disk)
    print(f'Synced {n_missing} new observation(s).')
else:
    print('Already up-to-date.')

print(f'X: {X_disk.shape} | Y: {Y_disk.shape}')
current_week = len(Y_disk) - INITIAL_SIZE + 1
print(f'Week {current_week} | {len(Y_disk)} total observations ({INITIAL_SIZE} initial + {len(Y_disk)-INITIAL_SIZE} added)')

Already up-to-date.
X: (45, 8) | Y: (45,)
Week 6 | 45 total observations (40 initial + 5 added)


In [2]:
# Cell 3: Load data and inspect -- Function 8: Portfolio Optimisation (8D), Maximise

X = np.load(DATA_PATH_X)
Y = np.load(DATA_PATH_Y)
n_dims = X.shape[1]

print(f'Input  shape : {X.shape}')
print(f'Output shape : {Y.shape}')
print()

pairs = sorted(zip(Y.tolist(), X.tolist()), key=lambda p: abs(p[0]), reverse=True)
Y_sorted = [p[0] for p in pairs]
X_sorted = [p[1] for p in pairs]

print('=' * 72)
print('  All observations (sorted descending by |Y|)')
print('=' * 72)
for i, (y_val, x_val) in enumerate(zip(Y_sorted, X_sorted)):
    marker = '  <-- best' if i == 0 else ''
    x_str = ', '.join(f'{v:.6f}' for v in x_val)
    print(f'  [{i+1:2d}]  X = [{x_str}]   Y = {y_val:+.6e}{marker}')
print('=' * 72)

best_Y = Y_sorted[0]
best_X = np.array(X_sorted[0])
best_x_str = ', '.join(f'{v:.8f}' for v in best_X)
print(f'\n  Best |Y*| = {abs(best_Y):.6e}  (Y* = {best_Y:.6e})')
print(f'  Best X*   = [{best_x_str}]')

Input  shape : (45, 8)
Output shape : (45,)

  All observations (sorted descending by |Y|)
  [ 1]  X = [0.056447, 0.065956, 0.022929, 0.038786, 0.403935, 0.801055, 0.488307, 0.893085]   Y = +9.598482e+00  <-- best
  [ 2]  X = [0.000000, 0.000000, 0.000000, 0.000000, 1.000000, 1.000000, 0.000000, 1.000000]   Y = +9.518300e+00
  [ 3]  X = [0.192640, 0.630677, 0.416796, 0.490529, 0.796086, 0.654567, 0.276241, 0.295518]   Y = +9.344274e+00
  [ 4]  X = [0.481245, 0.102461, 0.219486, 0.677322, 0.247509, 0.244341, 0.163825, 0.715962]   Y = +9.183005e+00
  [ 5]  X = [0.145120, 0.119328, 0.420888, 0.387609, 0.155423, 0.875172, 0.510560, 0.728611]   Y = +9.141639e+00
  [ 6]  X = [0.076274, 0.101214, 0.383035, 0.338493, 0.113685, 0.882235, 0.615428, 0.796463]   Y = +9.038246e+00
  [ 7]  X = [0.076274, 0.101214, 0.383035, 0.338493, 0.113685, 0.882235, 0.615428, 0.796463]   Y = +9.038246e+00
  [ 8]  X = [0.044329, 0.013581, 0.258198, 0.577644, 0.051280, 0.158563, 0.591030, 0.077953]   Y = +9.013075

In [3]:
# Cell 4: Fit GP -- raw Y (no log transform; F8 values range ~5 to 9.6, normal scale)
import warnings
warnings.filterwarnings('ignore')
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF

kernel = RBF(length_scale=0.3, length_scale_bounds='fixed')
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-4)
gp.fit(X, Y)

print('=' * 62)
print('  GP Fitting Results')
print('=' * 62)
print(f'  Kernel                  : {gp.kernel_}')
print(f'  Log-marginal-likelihood : {gp.log_marginal_likelihood_value_:.4f}')
pred_mean, pred_std = gp.predict(best_X.reshape(1, -1), return_std=True)
print('  Sanity check at best known X* (Y=9.598, initial data):')
print(f'    GP predicted mean = {pred_mean[0]:.4f}')
print(f'    Actual Y*         = {best_Y:.4f}')
print(f'    GP predicted std  = {pred_std[0]:.6f}')

w4_corner = np.array([0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 1.0])
pred_w4, std_w4 = gp.predict(w4_corner.reshape(1, -1), return_std=True)
print(f'  Sanity check at W4 corner [0,0,0,0,1,1,0,1] (Y=9.518):')
print(f'    GP predicted mean = {pred_w4[0]:.4f}')
print(f'    Actual Y (W4)     = 9.5183')
print(f'    GP predicted std  = {std_w4[0]:.6f}')
print('=' * 62)

  GP Fitting Results
  Kernel                  : RBF(length_scale=0.3)
  Log-marginal-likelihood : -978.3408
  Sanity check at best known X* (Y=9.598, initial data):
    GP predicted mean = 9.5977
    Actual Y*         = 9.5985
    GP predicted std  = 0.009999
  Sanity check at W4 corner [0,0,0,0,1,1,0,1] (Y=9.518):
    GP predicted mean = 9.5174
    Actual Y (W4)     = 9.5183
    GP predicted std  = 0.009999


In [4]:
# Cell 5: GP UCB Acquisition -- focused random search near W4 corner
# W4 at [0,0,0,0,1,1,0,1] scored 9.518 -- best weekly result.
# Pattern: x1,x2,x3,x4,x7 → 0 (low); x5,x6,x8 → 1 (high). F8 rewards extremes.
# Search ±0.10 around the corner -- if peak is slightly off-corner, this will find it.

w4_corner = np.array([0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 1.0])
np.random.seed(42)
X_grid = np.clip(
    w4_corner + np.random.uniform(-0.10, 0.10, size=(50000, n_dims)),
    0.0, 1.0
)

gp_mean, gp_std = gp.predict(X_grid, return_std=True)

beta = 1.0
ucb_scores = gp_mean + beta * gp_std

best_idx = np.argmax(ucb_scores)
next_x = X_grid[best_idx]
portal_string = '-'.join([f'{v:.6f}' for v in next_x])

print('=' * 72)
print('  GP UCB Acquisition (beta=1.0, W4 corner exploitation)')
print('=' * 72)
w4_str = ', '.join(f'{v:.4f}' for v in w4_corner)
print(f'  W4 corner anchor : [{w4_str}]')
print(f'  Search radius    : ±0.10  |  50,000 random points')
print(f'  Beta             : {beta}  (exploitation)')
next_str = ', '.join(f'{v:.6f}' for v in next_x)
print(f'  Next query       : [{next_str}]')
print(f'  UCB score        : {ucb_scores[best_idx]:.4f}')
print(f'  GP mean          : {gp_mean[best_idx]:.4f}')
print(f'  GP std           : {gp_std[best_idx]:.6f}')
print('=' * 72)

  GP UCB Acquisition (beta=1.0, W4 corner exploitation)
  W4 corner anchor : [0.0000, 0.0000, 0.0000, 0.0000, 1.0000, 1.0000, 0.0000, 1.0000]
  Search radius    : ±0.10  |  50,000 random points
  Beta             : 1.0  (exploitation)
  Next query       : [0.000000, 0.007711, 0.000000, 0.007233, 0.948719, 0.990493, 0.033594, 1.000000]
  UCB score        : 9.6773
  GP mean          : 9.4704
  GP std           : 0.206827


In [5]:
# Cell 6: Sanity check -- W4 corner pattern + distance check
# Expected: x1,x2,x3,x4,x7 < 0.15 (near zero); x5,x6,x8 > 0.85 (near one)

w4_corner = np.array([0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 1.0])
dist_to_w4 = np.linalg.norm(next_x - w4_corner)

# 0-indexed: low dims = 0,1,2,3,6 (x1,x2,x3,x4,x7); high dims = 4,5,7 (x5,x6,x8)
low_labels   = ['x1', 'x2', 'x3', 'x4', 'x7']
low_indices  = [0, 1, 2, 3, 6]
high_labels  = ['x5', 'x6', 'x8']
high_indices = [4, 5, 7]

violations_low  = [low_labels[i]  for i, idx in enumerate(low_indices)  if next_x[idx] >= 0.15]
violations_high = [high_labels[i] for i, idx in enumerate(high_indices) if next_x[idx] <= 0.85]

print('=' * 62)
print('  Sanity Check (W4 corner pattern: low/high dimensions)')
print('=' * 62)
next_str = ', '.join(f'{v:.6f}' for v in next_x)
print(f'  Proposed query : [{next_str}]')
print(f'  W4 corner      : [0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 1.0]')
print(f'  Distance to W4 : {dist_to_w4:.6f}')
print()
for lbl, idx in zip(low_labels, low_indices):
    ok = next_x[idx] < 0.15
    status = 'OK' if ok else f'WARNING: {next_x[idx]:.4f} >= 0.15'
    print(f'  {lbl} = {next_x[idx]:.6f}  (should be < 0.15)  [{status}]')
for lbl, idx in zip(high_labels, high_indices):
    ok = next_x[idx] > 0.85
    status = 'OK' if ok else f'WARNING: {next_x[idx]:.4f} <= 0.85'
    print(f'  {lbl} = {next_x[idx]:.6f}  (should be > 0.85)  [{status}]')
print()
if dist_to_w4 > 0.25:
    print(f'  WARNING: distance to W4 corner = {dist_to_w4:.4f} > 0.25 -- consider tightening radius.')
else:
    print(f'  OK: distance to W4 corner = {dist_to_w4:.4f} (within 0.25).')
if violations_low:
    print(f'  WARNING: {violations_low} are >= 0.15 (should be near zero).')
if violations_high:
    print(f'  WARNING: {violations_high} are <= 0.85 (should be near one).')
if not violations_low and not violations_high:
    print('  OK: all dimensions follow the expected W4 corner pattern.')
print('=' * 62)

  Sanity Check (W4 corner pattern: low/high dimensions)
  Proposed query : [0.000000, 0.007711, 0.000000, 0.007233, 0.948719, 0.990493, 0.033594, 1.000000]
  W4 corner      : [0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 1.0]
  Distance to W4 : 0.062932

  x1 = 0.000000  (should be < 0.15)  [OK]
  x2 = 0.007711  (should be < 0.15)  [OK]
  x3 = 0.000000  (should be < 0.15)  [OK]
  x4 = 0.007233  (should be < 0.15)  [OK]
  x7 = 0.033594  (should be < 0.15)  [OK]
  x5 = 0.948719  (should be > 0.85)  [OK]
  x6 = 0.990493  (should be > 0.85)  [OK]
  x8 = 1.000000  (should be > 0.85)  [OK]

  OK: distance to W4 corner = 0.0629 (within 0.25).
  OK: all dimensions follow the expected W4 corner pattern.


In [6]:
print('=' * 72)
print('  SUMMARY -- Week 6 Bayesian Optimisation')
print('=' * 72)
print('  Function   : 8 -- Portfolio Optimisation (8D)')
print('  Objective  : Maximise Y')
print(f'  Method     : GP UCB (beta=1.0, ±0.10 around W4 corner [0,0,0,0,1,1,0,1])')
print(f'  Kernel     : RBF(length_scale=0.3), alpha=1e-4, raw Y fit')
print()
best_x_s = ', '.join(f'{v:.6f}' for v in best_X)
next_s   = ', '.join(f'{v:.6f}' for v in next_x)
print(f'  Best X* (initial data) : [{best_x_s}]')
print(f'  Best Y* (initial data) : {best_Y:.6e}')
print(f'  W4 corner result       : 9.5183 (best weekly query)')
print()
print(f'  Next query  : [{next_s}]')
print()
print('  Portal submission string:')
print(f'  >>> {portal_string} <<<')
print('=' * 72)

  SUMMARY -- Week 6 Bayesian Optimisation
  Function   : 8 -- Portfolio Optimisation (8D)
  Objective  : Maximise Y
  Method     : GP UCB (beta=1.0, ±0.10 around W4 corner [0,0,0,0,1,1,0,1])
  Kernel     : RBF(length_scale=0.3), alpha=1e-4, raw Y fit

  Best X* (initial data) : [0.056447, 0.065956, 0.022929, 0.038786, 0.403935, 0.801055, 0.488307, 0.893085]
  Best Y* (initial data) : 9.598482e+00
  W4 corner result       : 9.5183 (best weekly query)

  Next query  : [0.000000, 0.007711, 0.000000, 0.007233, 0.948719, 0.990493, 0.033594, 1.000000]

  Portal submission string:
  >>> 0.000000-0.007711-0.000000-0.007233-0.948719-0.990493-0.033594-1.000000 <<<


## Week 6 -- Reflection

**Function**: F8 -- Portfolio Optimisation (8D)

### Strategy change from Week 5
- Removed NN surrogate and SVM: W5 SVM chose GP candidate which was a duplicate of W3 (same 9.038 result). Wasted query.
- GP UCB with trust region radius=0.15 around initial data best [0.056, 0.066, 0.023, 0.039, 0.404, 0.801, 0.488, 0.893].
- Best Y=9.598 has never been beaten. This is the region to exploit.
- 45 observations -- richest dataset among all functions; GP predictions are reliable here.

### Query
- **Input submitted**: *(fill in portal submission string)*
- **Output received**: *(fill in after result)*
- **Best so far**: *(update after result)*

### What this result taught us
*(fill in after receiving result)*

### Strategy for Week 7
*(fill in after reflecting on result)*